# 01 - Data loading and validation

Reads the instrument's non-standard BMP exports, recovers acquisition
metadata from the filenames, and writes the clean image stack that every
later stage consumes.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [ ]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
print("Raw data :", config.raw_dir)
print("Outputs  :", config.processed_dir)

## Parse the raw exports

The exporter writes a 54-byte BMP header claiming 64bpp, then interleaved
`uint16` planes. PIL and OpenCV both misread this, so the payload is parsed
directly from the bytes.

In [ ]:
from src.preprocessing.load_images import load_images_from_config

images = load_images_from_config(config)
for key, image in images.items():
    print(f"{image.shape}  max={image.max():8.0f}  {key}")

## Recover acquisition metadata

Each field is matched independently, so a filename missing one field still
yields the rest instead of failing.

In [ ]:
import pandas as pd

from src.preprocessing.metadata import (
    attach_shapes,
    build_metadata_from_config,
    metadata_to_frame,
)

metadata = attach_shapes(build_metadata_from_config(list(images), config), images)
metadata_to_frame(metadata)[
    ["polarity", "mass", "shots", "current", "dimensions", "shape"]
]

## Sanity checks and figures

In [ ]:
from src.preprocessing.stack_io import summary_statistics

summary_statistics(images)

In [ ]:
from src.viz import overview

overview.plot_raw_overview(images, config.figure_path("01_raw_overview.png"))
overview.plot_intensity_histograms(
    images, config.figure_path("01_intensity_histograms.png")
)
print("figures written")

## Save the clean stack

Stored as a keyed archive rather than one cube, because a dataset may mix
acquisition geometries that cannot share a single array.

In [ ]:
from src.preprocessing.stack_io import save_clean_stack

written = save_clean_stack(images, metadata, config.processed_dir)
for name, path in written.items():
    print(f"{name:10s} -> {path.name}")

Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage load
```